# Hybrid CNN-LSTM Stock Price Predictor Model

In [78]:
import pandas as pd
import numpy as np
from keras.layers import Input, Conv1D, Dense, BatchNormalization, Dropout, LSTM
from keras.models import Model

## Data

In [79]:
df = pd.read_csv("data/all_data.csv")
df.drop(columns=["date"], inplace=True)

# Create numerical representation of labels for model to use
labels = sorted(list(set(df["label"])))
labels_indices = dict((l, i) for i, l in enumerate(labels))
indices_labels = dict((i, l) for i, l in enumerate(labels))

# Down=0, Flat=1, Up=2  (alphabetical order)
df["label_num"] = df["label"].map(labels_indices)

In [80]:
X, y = [], []
features = df.drop(columns=["label", "label_num"]).values
label_nums = df["label_num"].values
window_size = 20
num_features = len(df.drop(columns=["label", "label_num"]).columns)

# Convert data into 3D for CNN (samples, timesteps, features)
for i in range(0, len(df) - window_size): 
        X.append(features[i : i + window_size]) # Add a 2D section of data of all features from the days within the current window size
        y.append(label_nums[i + window_size]) # Add the labels for each day in the window size

X = np.array(X)
y = np.array(y)

X.shape # Shape should be ((len(df) - window_size), window_size, num_features)
y.shape # Shape should be ((len(df) - window_size),)


(1187,)

In [81]:
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

## Model

All numbers are not real yet

In [82]:
inputs = Input(shape=(window_size,num_features))

# CNN Block
conv1 = Conv1D(filters=64, kernel_size=3, activation='relu')(inputs)
conv1 = BatchNormalization()(conv1)
conv1 = Dropout(0.2)(conv1)

conv2 = Conv1D(filters=64, kernel_size=5, activation='relu')(conv1)
conv2 = BatchNormalization()(conv2)
conv2 = Dropout(0.2)(conv2)

# LSTM Block
lstm = LSTM(5)(conv2)

# Dense layers and output
dense1 = Dense(5, activation='relu')(lstm)
outputs = Dense(3, activation='softmax')(dense1)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 20, 66)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_25 (Conv1D)              │ (None, 18, 64)         │        12,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_25          │ (None, 18, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_25 (Dropout)            │ (None, 18, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_26 (Conv1D)              │ (None, 14, 64)         │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_26          │ (None, 14, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 5)              │         1,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ (None, 3)              │            18 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,240 (137.66 KB)

 Trainable params: 34,984 (136.66 KB)

 Non-trainable params: 256 (1.00 KB)

## Training Model

In [85]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(X_train, y_train, batch_size=128, epochs=100)

score = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", score[0])
print("Test accuracy:", score[1])

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4478 - loss: 1.0525
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4563 - loss: 1.0360
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4668 - loss: 1.0223
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4658 - loss: 1.0245
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4647 - loss: 1.0142
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4742 - loss: 1.0276 
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4710 - loss: 1.0167 
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5047 - loss: 1.0052 
Epoch 9/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4900 - loss: 0.9974
Epoch 10/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4826 - loss: 1.0039
Epoch 11/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5121 - loss: 0.9850
Epoch 12/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5016 - loss: 0.98